# Train Vidgin (Tiny preset) on a free GPU

This notebook trains **your** Vidgin video engine — the PyTorch implementation in
[github.com/shahrozkalash/Vidgin](https://github.com/shahrozkalash/Vidgin) — on the Tiny preset:
a ~21.5M-parameter DiT + causal 3D VAE that learns to generate 17-frame, 64×64 clips of
Moving MNIST digits from text prompts like *"digit 3 moving left"*.

**Works on:** Google Colab (free T4) or Kaggle (free GPU, ~30 h/week).

**Before you run anything**
- **Colab:** `Runtime → Change runtime type → GPU (T4)`, then `Runtime → Run all`.
- **Kaggle:** `Settings → Accelerator → GPU`, Internet **on**, then `Run All`.

**Plan** (each training cell is resumable — if a session ends, see the last section):
1. Setup + GPU check
2. Train the VAE — 4,000 steps, ~1–2 h
3. Train the DiT — 15,000 steps, ~4–8 h
4. Generate a clip from your prompt and watch it right here

In [ ]:
# --- Setup: check the GPU ---
import torch
print("torch:", torch.__version__)
assert torch.cuda.is_available(), "No GPU found. Colab: Runtime > Change runtime type > GPU. Kaggle: Settings > Accelerator > GPU."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# --- Get the code (clones your repo; safe to re-run) ---
import os, subprocess
base = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
repo = os.path.join(base, "Vidgin")
if not os.path.isdir(repo):
    subprocess.run(["git", "clone", "https://github.com/shahrozkalash/Vidgin.git", repo], check=True)
else:
    subprocess.run(["git", "-C", repo, "pull", "--ff-only"], check=False)
os.chdir(repo)
print("Working in:", os.getcwd())

# torch is preinstalled on Colab/Kaggle; just make sure the small deps exist.
subprocess.run(["pip", "install", "-q", "numpy", "pytest", "pillow"], check=True)
print("Setup done.")

In [ ]:
# --- Optional sanity check: run the test suite (a few minutes) ---
# Proves the code on this machine is healthy before you spend GPU hours.
# Skip this cell if you just want to train.
!python -m pytest tests/ -q

## Step 1 — Train the VAE

The VAE learns to compress a 17-frame clip into a small latent space and rebuild it.
Watch the log: `loss` (recon + KL) should fall steadily. Checkpoints land in
`checkpoints/tiny-vae/` every 500 steps, so an interrupted session loses little.

In [ ]:
# --- Train the VAE: 4,000 steps ---
!python -m vidgen.train_vae --device cuda --steps 4000 --batch-size 16 --num-clips 20000 --lr 1e-4 --ckpt-dir checkpoints/tiny-vae --ckpt-every 500 --log checkpoints/tiny-vae/train.jsonl

In [ ]:
# --- VAE progress: first vs last logged loss ---
import json
rows = [json.loads(l) for l in open("checkpoints/tiny-vae/train.jsonl")]
print(f"steps logged: {len(rows)} | first loss: {rows[0]['loss']:.4f} | last loss: {rows[-1]['loss']:.4f}")
try:
    import matplotlib.pyplot as plt
    plt.plot([r["step"] for r in rows], [r["loss"] for r in rows])
    plt.xlabel("step"); plt.ylabel("VAE loss"); plt.title("VAE training"); plt.show()
except Exception as e:
    print("plot skipped:", e)

## Step 2 — Train the DiT (the video generator)

The DiT learns rectified flow in the VAE's latent space — this is the part that turns
a text prompt into motion. It's the long cell (4–8 h at full length). Kaggle sessions
cap at ~12 h and Colab free sessions vary, so if yours ends mid-way, **don't panic**:
jump to the resume commands at the bottom and continue — no progress is lost.

In [ ]:
# --- Train the DiT: 15,000 steps ---
!python -m vidgen.train --device cuda --steps 15000 --batch-size 16 --num-clips 20000 --lr 1e-4 --vae-ckpt checkpoints/tiny-vae/latest.pt --ckpt-dir checkpoints/tiny-dit --ckpt-every 1000 --log checkpoints/tiny-dit/train.jsonl

In [ ]:
# --- DiT progress: first vs last logged loss ---
import json
rows = [json.loads(l) for l in open("checkpoints/tiny-dit/train.jsonl")]
print(f"steps logged: {len(rows)} | first loss: {rows[0]['loss']:.4f} | last loss: {rows[-1]['loss']:.4f}")
try:
    import matplotlib.pyplot as plt
    plt.plot([r["step"] for r in rows], [r["loss"] for r in rows])
    plt.xlabel("step"); plt.ylabel("DiT flow loss"); plt.title("DiT training"); plt.show()
except Exception as e:
    print("plot skipped:", e)

## Step 3 — Generate a video

Edit the prompt below, then run the next two cells. Prompts the Tiny model understands:
digits and directions, e.g. *"digit 3 moving left"*, *"digit 7 moving right and digit 1 moving down"*.
Output: a 17-frame 64×64 clip, shown as an animated GIF right in the notebook.

In [ ]:
# --- Your prompt (edit me!) ---
PROMPT = "digit 3 moving left and digit 7 moving right"
print("Prompt:", PROMPT)

In [ ]:
# --- Generate the clip ---
!python -m vidgen.sample --device cuda --ckpt checkpoints/tiny-dit/latest.pt --vae-ckpt checkpoints/tiny-vae/latest.pt --prompt "$PROMPT" --steps 50 --method midpoint --outdir outputs/clip_final

In [ ]:
# --- Watch it: render the frames as an animated GIF ---
import numpy as np
from PIL import Image
from IPython.display import Image as ShowImage, display

a = np.load("outputs/clip_final/frames.npy")          # (17, 64, 64, 3), range about [-1, 1]
u = (np.clip(a, -1, 1) * 0.5 + 0.5)
u = (u * 255).astype("uint8")
frames = [Image.fromarray(u[i]).resize((256, 256), Image.NEAREST) for i in range(u.shape[0])]
frames[0].save("outputs/clip_final/clip.gif", save_all=True, append_images=frames[1:], duration=120, loop=0)
print("frames:", a.shape)
display(ShowImage("outputs/clip_final/clip.gif"))

## Session ended mid-training? Resume here

Re-run the **Setup** cells first (they re-clone and re-enter the folder; on Kaggle your
`checkpoints/` survive in the working directory — on Colab, re-upload or re-download them
first), then continue from the latest checkpoint. Only the remaining steps run again.

**Short on time?** Train fewer steps instead — VAE 2,000 / DiT 8,000 already shows moving
digits, just less crisp. You can always resume and add more steps later.

In [ ]:
# --- Resume commands (uncomment the stage you need) ---
# VAE, continue to 4,000 total steps:
#!python -m vidgen.train_vae --device cuda --resume checkpoints/tiny-vae/latest.pt --steps 4000 --batch-size 16 --num-clips 20000 --lr 1e-4 --ckpt-dir checkpoints/tiny-vae --ckpt-every 500 --log checkpoints/tiny-vae/train.jsonl
#
# DiT, continue to 15,000 total steps:
#!python -m vidgen.train --device cuda --resume checkpoints/tiny-dit/latest.pt --steps 15000 --batch-size 16 --num-clips 20000 --lr 1e-4 --vae-ckpt checkpoints/tiny-vae/latest.pt --ckpt-dir checkpoints/tiny-dit --ckpt-every 1000 --log checkpoints/tiny-dit/train.jsonl
print("Resume commands are above - remove the leading # from the one you need and run this cell.")

In [ ]:
# --- Save your trained model: zip the checkpoints ---
# Colab: the next line downloads the zip to your phone/computer.
# Kaggle: the zip appears in the notebook Output - download it from there.
import shutil, os
shutil.make_archive("vidgin_tiny_checkpoints", "zip", "checkpoints")
print("zipped:", os.path.getsize("vidgin_tiny_checkpoints.zip") // (1024*1024), "MB")
try:
    from google.colab import files
    files.download("vidgin_tiny_checkpoints.zip")
except ImportError:
    print("Not on Colab - grab vidgin_tiny_checkpoints.zip from the output/files panel.")

## What you have now

A Vidgin Tiny model **you trained**: `checkpoints/tiny-dit/latest.pt` (generator) +
`checkpoints/tiny-vae/latest.pt` (decoder). Keep the zip somewhere safe — with those two
files you can generate clips any time with `vidgen/sample.py`, no retraining needed.

Want more quality? Resume DiT training for another 10–15k steps, or raise `--num-clips`
for more digit/speed variety. Full notes live in the repo's `PYTHON_IMPLEMENTATION.md`.